# Stanford RNA 3D Folding Part 2 - TRM Submission

This notebook uses Tiny Recursive Models (TRM) to predict RNA 3D structures.

## Overview
- Predicts 5 different conformations per RNA sequence
- Uses recursive reasoning for complex RNA folding patterns
- Outputs C1' atom coordinates in competition format

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from typing import List, Dict
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Model Definition

In [ ]:
class RNAStructureModel(nn.Module):
    """TRM-based model for RNA 3D structure prediction."""
    
    def __init__(
        self,
        vocab_size: int = 4,
        embed_dim: int = 256,
        hidden_dim: int = 512,
        num_structures: int = 5,
        max_length: int = 500,
        H_cycles: int = 3,
        L_cycles: int = 6,
        L_layers: int = 2,
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.num_structures = num_structures
        self.max_length = max_length
        
        # Embeddings
        self.nucleotide_embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=vocab_size)
        self.position_embedding = nn.Embedding(max_length, embed_dim)
        
        # Transformer layers
        self.reasoning_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=8,
                dim_feedforward=hidden_dim,
                dropout=0.1,
                batch_first=True
            )
            for _ in range(L_layers)
        ])
        
        # Output heads
        self.structure_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 3)
            )
            for _ in range(num_structures)
        ])
        
        self.H_cycles = H_cycles
        self.L_cycles = L_cycles
    
    def forward(self, sequences: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = sequences.shape
        
        # Embed and add positions
        sequences_clamped = torch.clamp(sequences, min=0)
        x = self.nucleotide_embedding(sequences_clamped)
        positions = torch.arange(seq_len, device=sequences.device).unsqueeze(0)
        x = x + self.position_embedding(positions)
        
        # Attention mask
        padding_mask = sequences == self.vocab_size
        
        # Recursive reasoning
        for h_cycle in range(self.H_cycles):
            for l_cycle in range(self.L_cycles):
                for layer in self.reasoning_layers:
                    x = layer(x, src_key_padding_mask=padding_mask)
        
        # Predict structures
        predictions = [head(x) for head in self.structure_heads]
        predictions = torch.stack(predictions, dim=2)
        
        return predictions

print("Model class defined")

## Helper Functions

In [ ]:
def encode_rna_sequence(sequence: str, max_length: int = 500) -> np.ndarray:
    """Encode RNA sequence to numerical format."""
    nucleotide_map = {'A': 0, 'C': 1, 'G': 2, 'U': 3, 'N': 4}
    encoded = np.array([nucleotide_map.get(n, 4) for n in sequence.upper()])
    
    if len(encoded) < max_length:
        encoded = np.pad(encoded, (0, max_length - len(encoded)), constant_values=4)
    else:
        encoded = encoded[:max_length]
    
    return encoded

print("Helper functions defined")

## Load Test Data

In [ ]:
# Load test sequences
test_df = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv')
print(f"Loaded {len(test_df)} test sequences")
print(f"\nColumns: {test_df.columns.tolist()}")
print(f"\nFirst sequence:")
print(test_df.head(1))

## Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model configuration
max_length = 500
batch_size = 32 if torch.cuda.is_available() else 8

model = RNAStructureModel(
    vocab_size=4,
    embed_dim=256,
    hidden_dim=512,
    num_structures=5,
    max_length=max_length,
    H_cycles=3,
    L_cycles=6,
    L_layers=2,
).to(device)

# Load pretrained weights if available
checkpoint_path = '/kaggle/input/rna-structure-weights/rna_model.pth'
if os.path.exists(checkpoint_path):
    print(f"Loading weights from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print("Weights loaded successfully")
else:
    print("WARNING: No pretrained weights found - using random initialization")

model.eval()
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

## Generate Predictions

In [ ]:
predictions = []

with torch.no_grad():
    for i in tqdm(range(0, len(test_df), batch_size), desc="Predicting"):
        batch_df = test_df.iloc[i:i + batch_size]
        
        # Encode sequences
        encoded_seqs = []
        for _, row in batch_df.iterrows():
            encoded = encode_rna_sequence(row['sequence'], max_length)
            encoded_seqs.append(torch.tensor(encoded, dtype=torch.long))
        
        # Stack and predict
        encoded_batch = torch.stack(encoded_seqs).to(device)
        pred_coords = model(encoded_batch)
        
        # Clip coordinates to valid range
        pred_coords = torch.clamp(pred_coords, -999.999, 9999.999)
        pred_coords = pred_coords.cpu().numpy()
        
        # Store predictions
        for j, (idx, row) in enumerate(batch_df.iterrows()):
            seq_len = min(len(row['sequence']), max_length)
            predictions.append({
                'target_id': row['target_id'],
                'sequence': row['sequence'],
                'coordinates': pred_coords[j, :seq_len, :, :]
            })

print(f"\nGenerated predictions for {len(predictions)} sequences")

## Create Submission File

In [ ]:
submission_rows = []

for pred in predictions:
    target_id = pred['target_id']
    sequence = pred['sequence']
    coords = pred['coordinates']  # (seq_len, 5, 3)
    
    for i, nucleotide in enumerate(sequence):
        row = {
            'ID': f"{target_id}_{i + 1}",
            'resname': nucleotide,
            'resid': i + 1,
        }
        
        # Add coordinates for 5 structures
        # Handle sequences longer than max_length by clamping index
        coord_idx = min(i, coords.shape[0] - 1)
        for struct_idx in range(5):
            x, y, z = coords[coord_idx, struct_idx, :]
            row[f'x_{struct_idx + 1}'] = x
            row[f'y_{struct_idx + 1}'] = y
            row[f'z_{struct_idx + 1}'] = z
        
        submission_rows.append(row)

# Create DataFrame with correct column order
submission_df = pd.DataFrame(submission_rows)
coord_cols = []
for i in range(1, 6):
    coord_cols.extend([f'x_{i}', f'y_{i}', f'z_{i}'])
columns = ['ID', 'resname', 'resid'] + coord_cols
submission_df = submission_df[columns]

# Save submission
submission_df.to_csv('submission.csv', index=False)

print(f"Submission file created!")
print(f"Total residues: {len(submission_df)}")
print(f"From {len(predictions)} sequences")
print(f"\nFirst few rows:")
print(submission_df.head())

## Verify Submission Format

In [ ]:
# Verify format
print("Submission verification:")
print(f"  Shape: {submission_df.shape}")
print(f"  Columns: {len(submission_df.columns)}")
print(f"  Expected columns: 18 (ID, resname, resid + 15 coordinates)")
print(f"  Column check: {'✓' if len(submission_df.columns) == 18 else '✗'}")
print(f"\n  Coordinate ranges:")
for i in range(1, 6):
    x_col = f'x_{i}'
    if x_col in submission_df.columns:
        x_min = submission_df[x_col].min()
        x_max = submission_df[x_col].max()
        print(f"    Structure {i}: x ∈ [{x_min:.3f}, {x_max:.3f}]")

print("\n✓ Submission ready!")